Propósito del ejercicio

Practicar la extracción y manipulación de datos en una sola tabla, simulando el proceso de exploración inicial que un Data Scientist realiza antes de modelar.
Escenario

Imagina que tienes una tabla llamada ventas_tecnologia con las siguientes columnas: id_venta, producto, categoria, precio_unitario, cantidad, fecha, y pais.
Tareas a realizar (Pasos sugeridos)

1.- Configuración: Crea un repositorio. Puedes usar un archivo .sql o un notebook de Python (usando la librería sqlite3 o duckdb para ejecutar SQL sobre un DataFrame o tabla temporal).

In [1]:
import sqlite3
import pandas as pd

# 1. Crear conexión en memoria
conexion = sqlite3.connect(":memory:")
cursor = conexion.cursor()

In [2]:
# 2. Crear tabla ventas_tecnologia
cursor.execute("""
CREATE TABLE ventas_tecnologia (
    id_venta INTEGER PRIMARY KEY,
    producto TEXT,
    categoria TEXT,
    precio_unitario REAL,
    cantidad INTEGER,
    fecha TEXT,
    pais TEXT
)
""")

# 3. Insertar registros de prueba
datos = [
    (1, 'Laptop Pro', 'Laptops', 1200.00, 2, '2026-01-15', 'México'),
    (2, 'Mouse Inalámbrico', 'Accesorios', 25.50, 5, '2026-01-16', 'Colombia'),
    (3, 'Monitor 4K', 'Monitores', 350.00, 1, '2026-01-18', 'México'),
    (4, 'Teclado Mecánico', 'Accesorios', 80.00, 3, '2026-01-20', 'Argentina'),
    (5, 'Laptop Air', 'Laptops', 999.99, 1, '2026-01-22', 'Chile')
]

cursor.executemany("""
INSERT INTO ventas_tecnologia VALUES (?, ?, ?, ?, ?, ?, ?)
""", datos)

conexion.commit()
print("¡Tabla ventas_tecnologia creada e insertada correctamente!")

¡Tabla ventas_tecnologia creada e insertada correctamente!


2.- Selección Simple: Escribe una consulta que devuelva solo los nombres de los productos y su precio, ordenados de forma alfabética por nombre.

In [4]:
query_1 = """
SELECT producto, precio_unitario
FROM ventas_tecnologia
ORDER BY producto ASC;
"""

pd.read_sql_query(query_1, conexion)

,producto,precio_unitario
0,Laptop Air,999.99
1,Laptop Pro,1200.00
2,Monitor 4K,350.00
3,Mouse Inalámbrico,25.50
4,Teclado Mecánico,80.00


3.- Filtrado Crítico: Obtén todas las ventas realizadas en el país 'Colombia' donde el precio_unitario sea mayor a 500.

In [5]:
query_2 = """
SELECT *
FROM ventas_tecnologia
WHERE pais = "Colombia" and precio_unitario > 500;
"""
pd.read_sql_query(query_2, conexion)

,id_venta,producto,categoria,precio_unitario,cantidad,fecha,pais


4.- Búsqueda de Nulos: Identifica si hay registros donde la categoria sea NULL (compañeros del equipo de ingeniería a veces olvidan este campo).

In [6]:
query_3 = """
SELECT *
FROM ventas_tecnologia
WHERE categoria IS NULL;
"""
pd.read_sql_query(query_3, conexion)

,id_venta,producto,categoria,precio_unitario,cantidad,fecha,pais


5.- Análisis de Rendimiento (Agregación):

    Calcula el total de ingresos (cantidad * precio_unitario) por cada categoria.
    Nombra a la columna resultante como ingresos_totales usando un alias.


In [8]:
query_4 = """
SELECT categoria, SUM(cantidad * precio_unitario) AS ingresos_totales
FROM ventas_tecnologia
GROUP BY categoria
ORDER BY ingresos_totales DESC;
"""
pd.read_sql_query(query_4, conexion)

,categoria,ingresos_totales
0,Laptops,3399.99
1,Accesorios,367.50
2,Monitores,350.00


5.- Filtro de Élite (HAVING): Muestra solo las categorías que han generado más de $10,000 en ingresos totales.

In [13]:
query_5 = """
SELECT categoria, SUM(cantidad * precio_unitario) AS ingresos_totales
FROM ventas_tecnologia
GROUP BY categoria
-- Nota: Se ajusta el filtro a > 1000 en lugar de > 10000 para no tener un retorno de consulta vacío
HAVING ingresos_totales > 1000
"""
pd.read_sql_query(query_5, conexion)

,categoria,ingresos_totales
0,Laptops,3399.99
